# Main paper analysis

Run this notebook from the repository root. It rebuilds the paper-ready LLM tables and figures for the four current research questions.

Metric policy:
- zero-shot (`c0`) uses raw/open scoring;
- candidate-list settings (`c250`, `c500`, `c1000`, `cALL`) use strict candidate-aware scoring;
- strict scoring filters model predictions to the prompt-specific candidate list before computing metrics.


In [ ]:
from __future__ import annotations

import itertools
import json
import math
from collections.abc import Iterable, Sequence
from pathlib import Path
from typing import Any, cast

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display as ipy_display
from matplotlib.lines import Line2D
from scipy.stats import wilcoxon

from stability.metrics import recommendation_metrics
from stability.utils import bootstrap_mean_ci, canonicalize

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "data" / "output" / "paper"
FIGURE_DIR = OUTPUT_DIR / "figures"

DETERMINISTIC_PATHS = [
    ROOT / "data" / "output" / "evaluation_results_20260226_131313.parquet",
    ROOT / "data" / "output" / "evaluation_results_20260226_153912.parquet",
    ROOT / "data" / "output" / "evaluation_results_20260601_151356.parquet",
]

PROMPT_SOURCES = {
    "c0": {
        "path": ROOT / "data" / "processed" / "test_prompt_examples_c0_r10.jsonl",
        "n_candidates": "c0",
        "retriever": "Zero-shot",
        "candidate_label": "Zero-shot",
    },
    "c250": {
        "path": ROOT / "data" / "processed" / "test_prompt_examples_c250_r10.jsonl",
        "n_candidates": "c250",
        "retriever": "Semantic",
        "candidate_label": "250",
    },
    "c500": {
        "path": ROOT / "data" / "processed" / "test_prompt_examples_c500_r10.jsonl",
        "n_candidates": "c500",
        "retriever": "Semantic",
        "candidate_label": "500",
    },
    "c1000": {
        "path": ROOT / "data" / "processed" / "test_prompt_examples_c1000_r10.jsonl",
        "n_candidates": "c1000",
        "retriever": "Semantic",
        "candidate_label": "1000",
    },
    "cALL": {
        "path": ROOT / "data" / "processed" / "test_prompt_examples_cALL_r10.jsonl",
        "n_candidates": "cALL",
        "retriever": "Semantic",
        "candidate_label": "All",
    },
    "semantic_c250": {
        "path": ROOT / "data" / "processed" / "test_prompt_examples_c250_r10.jsonl",
        "n_candidates": "c250",
        "retriever": "Semantic",
        "candidate_label": "250",
    },
    "ease_c250": {
        "path": ROOT
        / "data"
        / "processed"
        / "test_prompt_examples_cf_EASE_c250_r10.jsonl",
        "n_candidates": "c250",
        "retriever": "EASE",
        "candidate_label": "250",
    },
    "sasrec_c250": {
        "path": ROOT
        / "data"
        / "processed"
        / "test_prompt_examples_seq_SASRec_c250_r10.jsonl",
        "n_candidates": "c250",
        "retriever": "SASRec",
        "candidate_label": "250",
    },
}

RQ3_PROMPT_SOURCE_ORDER = ["semantic_c250", "ease_c250", "sasrec_c250"]
RQ3_RERANKER_MODELS = ["claude-opus-4-6", "Llama-3.3-70B"]
RQ3_RESULT_GLOB = "evaluation_results_rq3_*.parquet"

RECBOLE_CF_PATH = ROOT / "data" / "output" / "evaluation_results_recbole_cf.parquet"
RECBOLE_SEQ_PATH = (
    ROOT / "data" / "output" / "evaluation_results_recbole_sequential.parquet"
)
RECBOLE_UNIFIED_SUMMARY_PATH = (
    ROOT / "data" / "recbole" / "evaluation_results" / "unified_summary.csv"
)

CATALOG_TITLE_PATH = ROOT / "data" / "recbole" / "redial" / "id_to_title.json"
TRAIN_INTERACTION_PATH = ROOT / "data" / "recbole" / "redial" / "redial.train.inter"

STABILITY_TRIAL_PATHS = [
    ROOT / "data" / "output" / "stability_results_20260602_155232.parquet",
    ROOT / "data" / "output" / "stability_results_20260602_231143.parquet",
]
STABILITY_AGGREGATE_PATHS = [
    ROOT / "data" / "output" / "stability_aggregated_20260602_155232.parquet",
    ROOT / "data" / "output" / "stability_aggregated_20260602_231143.parquet",
]

K_VALUES = [1, 5, 10]
METRIC_NAMES = ["hit_rate", "mrr", "precision", "recall", "f1", "ndcg"]
PRIMARY_METRIC = "paper_ndcg@10"
RNG = np.random.default_rng(20260603)

MODEL_DISPLAY_NAMES = {
    "infobip-gpt-4-1": "GPT-4.1",
    "infobip-gpt-4-1-mini": "GPT-4.1-mini",
    "gpt-5.2": "GPT-5.2",
    "claude-sonnet-4-6": "Claude Sonnet 4.6",
    "claude-opus-4-6": "Claude Opus 4.6",
}
MODEL_ORDER = [
    "Claude Opus 4.6",
    "GPT-5.2",
    "GPT-4.1",
    "Claude Sonnet 4.6",
    "GPT-4.1-mini",
    "Qwen2.5-7B-FT",
    "Gemma-2-9B",
    "Llama-3.3-70B",
    "Qwen2.5-7B",
    "Llama-3.1-8B",
    "Llama-3.2-3B",
]
CANDIDATE_ORDER = ["c0", "c250", "c500", "c1000", "cALL"]
CANDIDATE_LABELS = {
    "c0": "Zero-shot",
    "c250": "250",
    "c500": "500",
    "c1000": "1000",
    "cALL": "All",
}

# Paper figure styling: Okabe-Ito colorblind-safe palette and family groupings
# used by the RQ2 and RQ4 figures.
OKABE_ITO = [
    "#0072B2",  # blue
    "#D55E00",  # vermillion
    "#009E73",  # bluish green
    "#CC79A7",  # reddish purple
    "#E69F00",  # orange
    "#56B4E9",  # sky blue
]
PROPRIETARY_MODELS = [
    "Claude Opus 4.6",
    "GPT-5.2",
    "Claude Sonnet 4.6",
    "GPT-4.1",
    "GPT-4.1-mini",
]
OPEN_WEIGHT_MODELS = [
    # "Qwen2.5-7B-FT",
    "Llama-3.3-70B",
    "Gemma-2-9B",
    "Llama-3.1-8B",
    "Qwen2.5-7B",
    "Llama-3.2-3B",
]

## Helper functions

In [ ]:
def ensure_dirs() -> None:
    """Create output directories used by the paper analysis."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def display_model_name(model: str) -> str:
    """Return paper-facing model label for internal model identifiers."""
    return MODEL_DISPLAY_NAMES.get(model, model)


def sort_model_frame(df: pd.DataFrame, column: str = "model_display") -> pd.DataFrame:
    """Sort a dataframe by the fixed paper model order when possible."""
    order = {name: idx for idx, name in enumerate(MODEL_ORDER)}
    out = df.copy()
    out["_model_order"] = out[column].map(order).fillna(len(order)).astype(int)
    return out.sort_values(["_model_order", column]).drop(columns="_model_order")


def as_list(value: object) -> list[str]:
    """Convert parquet/list-like cells to a clean list of strings."""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    if isinstance(value, np.ndarray):
        return [str(item) for item in value.tolist() if item is not None]
    if isinstance(value, (list, tuple)):
        return [str(item) for item in value if item is not None]
    if isinstance(value, str):
        stripped = value.strip()
        if not stripped:
            return []
        if stripped.startswith("[") and stripped.endswith("]"):
            try:
                parsed = json.loads(stripped)
            except json.JSONDecodeError:
                return [stripped]
            return as_list(parsed)
        return [stripped]
    return [str(value)]


def canonical_topk(items: Sequence[str], k: int = 10) -> list[str]:
    """Return canonicalized, non-empty top-k item labels."""
    canon: list[str] = []
    for item in items[:k]:
        value = canonicalize(item)
        if value:
            canon.append(value)
    return canon


def extract_candidates_from_prompt(record: dict[str, Any]) -> list[str]:
    """Extract the actual candidate title block from a prompt JSONL record."""
    messages = record.get("messages", [])
    text = "\n".join(str(message.get("content", "")) for message in messages)
    close_idx = text.find("</CANDIDATES>")
    if close_idx < 0:
        return []
    # The instructions mention the literal <CANDIDATES> tag, so use the final
    # opening tag before the closing tag rather than the first occurrence.
    open_idx = text.rfind("<CANDIDATES>", 0, close_idx)
    if open_idx < 0:
        return []
    start_idx = open_idx + len("<CANDIDATES>")
    candidate_block = text[start_idx:close_idx]
    return [line.strip() for line in candidate_block.splitlines() if line.strip()]


def load_catalog_info() -> dict[str, Any]:
    """Load catalog titles and training popularity counts for beyond-accuracy metrics."""
    id_to_title_raw = json.loads(CATALOG_TITLE_PATH.read_text())
    id_to_title = {
        str(item_id): str(title) for item_id, title in id_to_title_raw.items()
    }
    canonical_to_title = {canonicalize(title): title for title in id_to_title.values()}

    train = pd.read_csv(TRAIN_INTERACTION_PATH, sep="\t")
    item_col = "item_id:token"
    counts_by_id = train[item_col].astype(str).value_counts().to_dict()
    popularity_by_title = {
        id_to_title[item_id]: int(count)
        for item_id, count in counts_by_id.items()
        if item_id in id_to_title
    }
    popularity_by_canonical = {
        canonicalize(title): count for title, count in popularity_by_title.items()
    }

    return {
        "id_to_title": id_to_title,
        "catalog_titles": list(id_to_title.values()),
        "catalog_set": frozenset(canonical_to_title),
        "catalog_size": len(id_to_title),
        "popularity_by_canonical": popularity_by_canonical,
    }


def recommendation_set_metrics(
    rows: Iterable[Sequence[str]],
    catalog_size: int,
    popularity_by_canonical: dict[str | None, int],
    k: int,
) -> dict[str, float]:
    """Compute RecBole-style ItemCoverage@K and AveragePopularity@K."""
    unique_items: set[str | None] = set()
    per_user_popularity: list[float] = []
    matched_counts: list[int] = []

    for items in rows:
        topk = canonical_topk(items, k=k)
        matched = [item for item in topk if item in popularity_by_canonical]
        unique_items.update(matched)
        matched_counts.append(len(matched))
        if matched:
            per_user_popularity.append(
                float(np.mean([popularity_by_canonical[item] for item in matched]))
            )
        else:
            per_user_popularity.append(np.nan)

    finite_popularity = np.array(per_user_popularity, dtype=float)
    finite_popularity = finite_popularity[np.isfinite(finite_popularity)]
    avg_popularity = (
        float(finite_popularity.mean()) if len(finite_popularity) else np.nan
    )

    return {
        f"item_coverage@{k}": len(unique_items) / catalog_size
        if catalog_size
        else np.nan,
        f"unique_items@{k}": float(len(unique_items)),
        f"avg_popularity@{k}": avg_popularity,
        f"catalog_match_rate@{k}": float(np.mean(np.array(matched_counts) / k))
        if matched_counts
        else np.nan,
    }


def load_prompt_info() -> dict[str, pd.DataFrame]:
    """Load available prompt JSONL files and candidate sets used in the analysis."""
    prompt_info: dict[str, pd.DataFrame] = {}
    for source, meta in PROMPT_SOURCES.items():
        path = Path(meta["path"])
        if not path.exists():
            print(
                f"Skipping missing prompt file for {source}: {path.relative_to(ROOT)}"
            )
            continue
        rows: list[dict[str, Any]] = []
        with path.open() as f:
            for prompt_idx, line in enumerate(f):
                record = json.loads(line)
                candidates = extract_candidates_from_prompt(record)
                rows.append(
                    {
                        "prompt_idx": prompt_idx,
                        "prompt_source": source,
                        "n_candidates": meta["n_candidates"],
                        "retriever": meta["retriever"],
                        "ground_truth_from_prompt": record.get("ground_truth", []),
                        "candidate_titles": candidates,
                        "candidate_set": frozenset(
                            canonicalize(candidate) for candidate in candidates
                        ),
                        "candidate_count": len(candidates),
                    }
                )
        prompt_info[source] = pd.DataFrame(rows)
    return prompt_info


def infer_prompt_source(
    row: dict[str, Any], fixed_prompt_source: str | None = None
) -> str:
    """Infer which prompt/candidate file a row should be scored against."""
    if fixed_prompt_source is not None:
        return fixed_prompt_source
    value = row.get("prompt_source")
    if isinstance(value, str) and value in PROMPT_SOURCES:
        return value
    retriever = str(row.get("retriever", "")).lower()
    if retriever == "ease":
        return "ease_c250"
    if retriever == "sasrec":
        return "sasrec_c250"
    n_candidates = str(row.get("n_candidates", "c250"))
    return n_candidates if n_candidates in PROMPT_SOURCES else "c250"


def candidate_set_for(
    prompt_info: dict[str, pd.DataFrame],
    prompt_source: str,
    prompt_idx: int,
) -> frozenset[str | None]:
    """Return canonical candidate set for a prompt source and prompt index."""
    if PROMPT_SOURCES[prompt_source]["n_candidates"] == "c0":
        return frozenset()
    df = prompt_info[prompt_source]
    return df.iloc[int(prompt_idx)]["candidate_set"]


def strict_filter_predictions(
    predictions: Sequence[str],
    candidate_set: frozenset[str | None],
) -> list[str]:
    """Keep only predictions that are present in the canonical candidate set."""
    filtered: list[str] = []
    seen: set[str | None] = set()
    for prediction in predictions:
        pred_canon = canonicalize(prediction)
        if pred_canon in candidate_set and pred_canon not in seen:
            filtered.append(prediction)
            seen.add(pred_canon)
    return filtered


def compute_metric_columns(
    predictions: Sequence[str],
    ground_truth: Sequence[str],
    prefix: str,
) -> dict[str, float]:
    """Compute recommendation metrics and prefix their column names."""
    metrics = recommendation_metrics(list(predictions), list(ground_truth), K_VALUES)
    return {f"{prefix}_{key}": float(value) for key, value in metrics.items()}


def score_rows(
    df: pd.DataFrame,
    prompt_info: dict[str, pd.DataFrame],
    *,
    catalog_info: dict[str, Any] | None = None,
    fixed_prompt_source: str | None = None,
) -> pd.DataFrame:
    """Add raw, strict, and paper-selected metric columns to evaluation rows."""
    records: list[dict[str, Any]] = []
    for row in df.to_dict("records"):
        row_dict = dict(row)
        prompt_source = infer_prompt_source(row_dict, fixed_prompt_source)
        n_candidates = str(PROMPT_SOURCES[prompt_source]["n_candidates"])
        prompt_idx = int(row_dict["prompt_idx"])
        predictions = as_list(row_dict["pred_items"])
        ground_truth = as_list(row_dict["ground_truth"])

        raw_metrics = compute_metric_columns(predictions, ground_truth, "raw")
        if n_candidates == "c0":
            strict_predictions = list(predictions)
        else:
            cset = candidate_set_for(prompt_info, prompt_source, prompt_idx)
            strict_predictions = strict_filter_predictions(predictions, cset)
        strict_metrics = compute_metric_columns(
            strict_predictions, ground_truth, "strict"
        )

        paper_source = raw_metrics if n_candidates == "c0" else strict_metrics
        paper_metrics = {
            key.replace("raw_", "paper_").replace("strict_", "paper_"): value
            for key, value in paper_source.items()
        }

        if catalog_info is None:
            raw_catalog_matches = []
            strict_catalog_matches = []
        else:
            catalog_set = catalog_info["catalog_set"]
            raw_catalog_matches = [
                item for item in canonical_topk(predictions) if item in catalog_set
            ]
            strict_catalog_matches = [
                item
                for item in canonical_topk(strict_predictions)
                if item in catalog_set
            ]

        row_dict.update(raw_metrics)
        row_dict.update(strict_metrics)
        row_dict.update(paper_metrics)
        row_dict.update(
            {
                "prompt_source": prompt_source,
                "retriever": PROMPT_SOURCES[prompt_source]["retriever"],
                "n_candidates": n_candidates,
                "strict_pred_items": strict_predictions,
                "num_strict_pred_items": len(strict_predictions),
                "candidate_valid_count@10": len(strict_predictions[:10]),
                "candidate_valid_rate@10": len(strict_predictions[:10]) / 10.0,
                "raw_pred_count@10": min(len(predictions[:10]), 10),
                "raw_catalog_match_count@10": len(raw_catalog_matches),
                "strict_catalog_match_count@10": len(strict_catalog_matches),
                "model_display": display_model_name(str(row_dict["model"])),
            }
        )
        records.append(row_dict)
    return pd.DataFrame.from_records(records)


def summarize_evaluation(
    df: pd.DataFrame,
    group_cols: Sequence[str],
    *,
    catalog_info: dict[str, Any] | None = None,
    include_candidate_validity: bool = True,
) -> pd.DataFrame:
    """Aggregate row-level metric columns by model/candidate groups."""
    rows: list[dict[str, Any]] = []
    metric_columns = [
        f"{prefix}_{metric}@{k}"
        for prefix in ["raw", "strict", "paper"]
        for metric in METRIC_NAMES
        for k in K_VALUES
    ]
    for group_key, group in df.groupby(list(group_cols), dropna=False):
        group_key_tuple = group_key if isinstance(group_key, tuple) else (group_key,)
        record = dict(zip(group_cols, group_key_tuple, strict=True))
        record["n_prompts"] = len(group)
        record["mean_num_pred_items"] = float(group["num_pred_items"].mean())
        record["mean_num_strict_pred_items"] = float(
            group["num_strict_pred_items"].mean()
        )
        if include_candidate_validity:
            record["candidate_valid_rate@10"] = float(
                group["candidate_valid_rate@10"].mean()
            )
        if catalog_info is not None:
            paper_item_rows = [
                as_list(preds) if n_candidates == "c0" else as_list(strict_preds)
                for preds, strict_preds, n_candidates in zip(
                    group["pred_items"],
                    group["strict_pred_items"],
                    group["n_candidates"],
                    strict=True,
                )
            ]
            for prefix, item_rows in {
                "raw": group["pred_items"],
                "strict": group["strict_pred_items"],
                "paper": paper_item_rows,
            }.items():
                for k in K_VALUES:
                    set_metrics = recommendation_set_metrics(
                        item_rows,
                        catalog_info["catalog_size"],
                        catalog_info["popularity_by_canonical"],
                        k,
                    )
                    for metric_name, value in set_metrics.items():
                        record[f"{prefix}_{metric_name}"] = value
        for col in metric_columns:
            record[col] = float(group[col].mean())
        mean, low, high = bootstrap_mean_ci(
            group[PRIMARY_METRIC].to_numpy(dtype=float), RNG, n_boot=2000
        )
        record[f"{PRIMARY_METRIC}_mean"] = mean
        record[f"{PRIMARY_METRIC}_ci_low"] = low
        record[f"{PRIMARY_METRIC}_ci_high"] = high
        rows.append(record)
    return pd.DataFrame.from_records(rows)


def load_deterministic_llm() -> pd.DataFrame:
    """Load and validate the final deterministic LLM dataframe."""
    parts = []
    for path in DETERMINISTIC_PATHS:
        df = pd.read_parquet(path)
        df["source_file"] = path.name
        parts.append(df)

    llm = pd.concat(parts, ignore_index=True)
    old_bad_gpt52_c0 = (
        llm["model"].eq("gpt-5.2")
        & llm["n_candidates"].eq("c0")
        & llm["source_file"].eq("evaluation_results_20260226_153912.parquet")
    )
    llm = llm.loc[~old_bad_gpt52_c0].copy()

    assert len(llm) == 40959, f"Unexpected deterministic row count: {len(llm)}"
    duplicate_keys = llm.duplicated(["model", "n_candidates", "prompt_idx"])
    assert not duplicate_keys.any(), (
        "Duplicate deterministic model/candidate/prompt rows"
    )
    assert not llm["model"].str.contains("gemini", case=False, na=False).any()
    return llm

In [ ]:
def holm_adjust(p_values: dict[str, float]) -> dict[str, float]:
    """Return Holm-adjusted p-values for a set of hypotheses."""
    items = sorted(p_values.items(), key=lambda item: item[1])
    adjusted: dict[str, float] = {}
    running_max = 0.0
    m = len(items)
    for rank, (key, p_value) in enumerate(items):
        adj = min(1.0, (m - rank) * p_value)
        running_max = max(running_max, adj)
        adjusted[key] = running_max
    return adjusted


def wilcoxon_vs_ease(
    llm_scored: pd.DataFrame,
    setting: str,
    *,
    alpha: float = 0.05,
) -> pd.DataFrame:
    """Paired Wilcoxon tests comparing each LLM to EASE@semantic250."""
    ease = pd.read_parquet(RECBOLE_CF_PATH)
    ease = ease.loc[
        ease["model"].eq("EASE") & ease["n_candidates"].eq("c250"),
        ["prompt_idx", "ndcg@10"],
    ].copy()
    # Some RecBole export rows carry prompt_idx=-1 for every row. The rows are
    # nevertheless in test-prompt order, so restore the paired index by row order.
    if ease["prompt_idx"].nunique(dropna=False) != len(ease):
        ease["prompt_idx"] = np.arange(len(ease))
    ease = ease.rename(columns={"ndcg@10": "ease_ndcg@10"})

    score_col = "paper_ndcg@10"
    p_values: dict[str, float] = {}
    raw_stats: dict[str, dict[str, float]] = {}
    for model, group in llm_scored.loc[llm_scored["n_candidates"].eq(setting)].groupby(
        "model"
    ):
        paired = group[["prompt_idx", score_col]].merge(
            ease, on="prompt_idx", how="inner"
        )
        diff = paired[score_col].to_numpy(dtype=float) - paired[
            "ease_ndcg@10"
        ].to_numpy(dtype=float)
        if np.allclose(diff, 0.0):
            statistic = 0.0
            p_value = 1.0
        else:
            result = wilcoxon(diff, alternative="greater", zero_method="wilcox")
            statistic = float(result.statistic)
            p_value = float(result.pvalue)
        p_values[model] = p_value
        raw_stats[model] = {
            "wilcoxon_statistic": statistic,
            "p_raw": p_value,
            "mean_delta_vs_ease": float(np.mean(diff)),
            "n_pairs": float(len(paired)),
        }

    adjusted = holm_adjust(p_values)
    rows = []
    for model, values in raw_stats.items():
        rows.append(
            {
                "model": model,
                **values,
                "p_holm": adjusted[model],
                "significant_vs_ease": adjusted[model] < alpha,
            }
        )
    return pd.DataFrame.from_records(rows)


def format_metric(value: float, *, star: bool = False, digits: int = 4) -> str:
    """Format a table metric, optionally appending the significance marker."""
    if pd.isna(value):
        return "—"
    suffix = "∗" if star else ""
    return f"{value:.{digits}f}{suffix}"


def build_rq1_table(
    deterministic_summary: pd.DataFrame,
    significance: pd.DataFrame,
    setting: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Build the requested LLM table for one setting."""
    sig_map = significance.set_index("model")["significant_vs_ease"].to_dict()
    p_map = significance.set_index("model")["p_holm"].to_dict()
    rows = []
    stats_rows = []
    subset = deterministic_summary.loc[
        deterministic_summary["n_candidates"].eq(setting)
    ].copy()
    subset = subset.sort_values(PRIMARY_METRIC, ascending=False)
    for row in subset.to_dict("records"):
        model = str(row["model"])
        star = bool(sig_map.get(model, False))
        rows.append(
            {
                "Reranker model": row["model_display"],
                "NDCG@10": format_metric(row["paper_ndcg@10"], star=star),
                "Hit@10": format_metric(row["paper_hit_rate@10"]),
                "Cov.@10": format_metric(row["paper_item_coverage@10"]),
                "Pop.@10": format_metric(row["paper_avg_popularity@10"], digits=2),
            }
        )
        stats_rows.append(
            {
                "model": model,
                "model_display": row["model_display"],
                "setting": setting,
                "paper_ndcg@10": row["paper_ndcg@10"],
                "paper_hit_rate@10": row["paper_hit_rate@10"],
                "paper_item_coverage@10": row["paper_item_coverage@10"],
                "paper_avg_popularity@10": row["paper_avg_popularity@10"],
                "p_holm_vs_ease": p_map.get(model, np.nan),
                "significant_vs_ease": star,
            }
        )
    return pd.DataFrame.from_records(rows), pd.DataFrame.from_records(stats_rows)


def build_rq2_table(deterministic_summary: pd.DataFrame) -> pd.DataFrame:
    """Build candidate-pool sensitivity table for all LLMs."""
    table = deterministic_summary[
        [
            "model",
            "model_display",
            "n_candidates",
            PRIMARY_METRIC,
            "raw_ndcg@10",
            "strict_ndcg@10",
        ]
    ].copy()
    table["Candidate pool"] = table["n_candidates"].map(CANDIDATE_LABELS)
    table["metric_policy"] = np.where(
        table["n_candidates"].eq("c0"),
        "raw/open",
        "strict candidate-aware",
    )
    table["candidate_order"] = table["n_candidates"].map(
        {value: idx for idx, value in enumerate(CANDIDATE_ORDER)}
    )
    return sort_model_frame(table)


# Per-model markers used in the RQ2 figure (cycled within each model class).
RQ2_MARKERS = ["o", "s", "^", "D", "v", "P"]
# Display labels for candidate pool sizes (paper convention: c0..cALL).
RQ2_CANDIDATE_DISPLAY = {
    "Zero-shot": "c0",
    "250": "c250",
    "500": "c500",
    "1000": "c1000",
    "All": "cAll",
}
RQ2_CANDIDATE_ORDER = ["c0", "c250", "c500", "c1000", "cAll"]


def save_rq2_line_figure(figure_table: pd.DataFrame) -> None:
    """Save RQ2 line figure: NDCG@10 vs candidate pool size.

    Proprietary models use solid lines and open-weight models use dashed
    lines; within each class color and marker shape cycle so every model has
    a unique combination. Legends sit to the right of the axes so the figure
    stays short on the page when scaled to a column width.
    """
    df = sort_model_frame(figure_table).copy()
    df["candidate_label"] = df["Candidate pool"].map(RQ2_CANDIDATE_DISPLAY)
    df["candidate_label"] = pd.Categorical(
        df["candidate_label"], categories=RQ2_CANDIDATE_ORDER, ordered=True
    )
    df = df.sort_values(["model_display", "candidate_label"])

    fig, ax = plt.subplots(figsize=(4.6, 2.2))

    def _plot_group(models, linestyle, *, hollow):
        for i, model in enumerate(models):
            sub = df.loc[df["model_display"].eq(model)].sort_values("candidate_label")
            if sub.empty:
                continue
            color = OKABE_ITO[i % len(OKABE_ITO)]
            ax.plot(
                sub["candidate_label"].astype(str),
                sub[PRIMARY_METRIC],
                color=color,
                linestyle=linestyle,
                marker=RQ2_MARKERS[i % len(RQ2_MARKERS)],
                markerfacecolor="white" if hollow else color,
                markeredgecolor=color,
                markeredgewidth=1.0,
                label=model,
            )

    _plot_group(PROPRIETARY_MODELS, linestyle="-", hollow=False)
    _plot_group(OPEN_WEIGHT_MODELS, linestyle="--", hollow=True)

    ax.set_xlabel("Candidate pool")
    ax.set_ylabel("NDCG@10")
    ax.set_ylim(0.0, 0.32)
    _minimal_axes(ax)

    prop_handles = [
        Line2D(
            [],
            [],
            color=OKABE_ITO[i % len(OKABE_ITO)],
            linestyle="-",
            marker=RQ2_MARKERS[i % len(RQ2_MARKERS)],
            label=m,
        )
        for i, m in enumerate(PROPRIETARY_MODELS)
    ]
    open_handles = [
        Line2D(
            [],
            [],
            color=OKABE_ITO[i % len(OKABE_ITO)],
            linestyle="--",
            marker=RQ2_MARKERS[i % len(RQ2_MARKERS)],
            markerfacecolor="white",
            markeredgecolor=OKABE_ITO[i % len(OKABE_ITO)],
            markeredgewidth=1.0,
            label=m,
        )
        for i, m in enumerate(OPEN_WEIGHT_MODELS)
    ]
    leg1 = ax.legend(
        handles=prop_handles,
        title=r"\textit{Proprietary}",
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        ncol=1,
        borderpad=0.2,
    )
    leg1._legend_box.align = "left"
    leg2 = ax.legend(
        handles=open_handles,
        title=r"\textit{Open-weight}",
        loc="lower left",
        bbox_to_anchor=(1.02, 0.0),
        ncol=1,
        borderpad=0.2,
    )
    leg2._legend_box.align = "left"
    ax.add_artist(leg1)

    # Reserve room on the right for the two stacked legends so tight_layout
    # does not clip them.
    fig.tight_layout(rect=(0.0, 0.0, 0.72, 1.0))
    ipy_display(fig)
    for ext in ["png", "pdf"]:
        fig.savefig(
            FIGURE_DIR / f"figure_rq2_candidate_pool_ndcg.{ext}",
            bbox_extra_artists=(leg1, leg2),
        )
    plt.close(fig)

In [ ]:
def candidate_oracle_metrics(prompt_info: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Compute candidate recall and oracle NDCG for RQ3 prompt sources."""
    rows = []
    for source in RQ3_PROMPT_SOURCE_ORDER:
        meta = PROMPT_SOURCES[source]
        df = prompt_info[source]
        recalls: list[float] = []
        oracle_ndcgs: list[float] = []
        for row in df.itertuples(index=False):
            ground_truth = as_list(row.ground_truth_from_prompt)
            truth_canon = [canonicalize(item) for item in ground_truth if item]
            candidate_titles = as_list(row.candidate_titles)
            candidate_set = set(row.candidate_set)
            relevant = [
                item for item in ground_truth if canonicalize(item) in candidate_set
            ]
            recalls.append(len(relevant) / len(truth_canon) if truth_canon else np.nan)

            relevant_set = {canonicalize(item) for item in relevant}
            oracle_predictions = relevant + [
                item
                for item in candidate_titles
                if canonicalize(item) not in relevant_set
            ]
            oracle_ndcgs.append(
                recommendation_metrics(oracle_predictions, ground_truth, [10])[
                    "ndcg@10"
                ]
            )

        rows.append(
            {
                "prompt_source": source,
                "Retriever": meta["retriever"],
                "CandR@250": float(np.nanmean(recalls)),
                "Oracle NDCG@10": float(np.nanmean(oracle_ndcgs)),
                "n_prompts": len(df),
            }
        )
    return pd.DataFrame.from_records(rows)


def load_optional_rq3_results() -> pd.DataFrame:
    """Load optional RQ3 LLM reranking outputs if they have been generated."""
    paths = sorted((ROOT / "data" / "output").glob(RQ3_RESULT_GLOB))
    if not paths:
        return pd.DataFrame()
    parts = []
    for path in paths:
        df = pd.read_parquet(path)
        df["source_file"] = path.name
        parts.append(df)
    out = pd.concat(parts, ignore_index=True)
    assert not out["model"].str.contains("gemini", case=False, na=False).any()
    return out


def build_rq3_table(
    prompt_info: dict[str, pd.DataFrame],
    deterministic_summary: pd.DataFrame,
    catalog_info: dict[str, Any],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Build RQ3 retriever table for every configured LLM reranker."""
    candidate_table = candidate_oracle_metrics(prompt_info)
    rq3_results = load_optional_rq3_results()
    scored_rq3 = pd.DataFrame()
    summary_rq3 = pd.DataFrame()

    ndcg_by_model_source: dict[tuple[str, str], float] = {}
    for model in RQ3_RERANKER_MODELS:
        semantic = deterministic_summary.loc[
            deterministic_summary["model"].eq(model)
            & deterministic_summary["n_candidates"].eq("c250")
        ]
        if not semantic.empty:
            ndcg_by_model_source[(model, "semantic_c250")] = float(
                semantic.iloc[0][PRIMARY_METRIC]
            )

    if not rq3_results.empty:
        scored_rq3 = score_rows(rq3_results, prompt_info, catalog_info=catalog_info)
        scored_rq3.to_parquet(
            OUTPUT_DIR / "rq3_retriever_llm_scored.parquet", index=False
        )
        summary_rq3 = summarize_evaluation(
            scored_rq3,
            [
                "model",
                "model_display",
                "model_type",
                "prompt_source",
                "retriever",
                "n_candidates",
            ],
            catalog_info=catalog_info,
        )
        summary_rq3.to_csv(OUTPUT_DIR / "rq3_retriever_llm_summary.csv", index=False)
        summary_rq3.to_parquet(
            OUTPUT_DIR / "rq3_retriever_llm_summary.parquet", index=False
        )
        configured = summary_rq3.loc[summary_rq3["model"].isin(RQ3_RERANKER_MODELS)]
        for row in configured.to_dict("records"):
            ndcg_by_model_source[(row["model"], row["prompt_source"])] = row[
                "paper_ndcg@10"
            ]

    long_rows = []
    for candidate_row in candidate_table.to_dict("records"):
        prompt_source = candidate_row["prompt_source"]
        for model in RQ3_RERANKER_MODELS:
            long_rows.append(
                {
                    "Retriever": candidate_row["Retriever"],
                    "Reranker model": display_model_name(model),
                    "CandR@250": candidate_row["CandR@250"],
                    "Oracle NDCG@10": candidate_row["Oracle NDCG@10"],
                    "NDCG@10": ndcg_by_model_source.get((model, prompt_source), np.nan),
                }
            )
    long_table = pd.DataFrame.from_records(long_rows)
    long_table.to_csv(OUTPUT_DIR / "table_rq3_retriever_effect_long.csv", index=False)

    wide = candidate_table[["Retriever", "CandR@250", "Oracle NDCG@10"]].copy()
    for model in RQ3_RERANKER_MODELS:
        col = f"{display_model_name(model)} NDCG@10"
        wide[col] = candidate_table["prompt_source"].map(
            {
                source: ndcg_by_model_source.get((model, source), np.nan)
                for source in RQ3_PROMPT_SOURCE_ORDER
            }
        )
    wide.to_csv(OUTPUT_DIR / "table_rq3_retriever_effect.csv", index=False)
    wide.to_latex(
        OUTPUT_DIR / "table_rq3_retriever_effect.tex", index=False, float_format="%.4f"
    )
    return wide, summary_rq3

In [ ]:
def summarize_recbole_parquet(
    path: Path,
    model_type: str,
    catalog_info: dict[str, Any] | None = None,
) -> pd.DataFrame:
    """Aggregate RecBole candidate-reranking per-user parquet results."""
    df = pd.read_parquet(path)
    rows = []
    for group_key, group in df.groupby(["model", "n_candidates"]):
        model, n_candidates = cast("tuple[str, str]", group_key)
        record: dict[str, Any] = {
            "model_type": model_type,
            "model": model,
            "eval_method": "candidate_reranking",
            "n_candidates": n_candidates,
            "n_prompts": len(group),
        }
        if catalog_info is not None:
            for k in K_VALUES:
                record.update(
                    recommendation_set_metrics(
                        group["pred_items"],
                        catalog_info["catalog_size"],
                        catalog_info["popularity_by_canonical"],
                        k,
                    )
                )
        for metric in METRIC_NAMES:
            for k in K_VALUES:
                col = f"{metric}@{k}"
                record[col] = float(group[col].mean())
        rows.append(record)
    return pd.DataFrame.from_records(rows)


def load_recbole_summaries(catalog_info: dict[str, Any]) -> pd.DataFrame:
    """Write compact RecBole candidate-reranking summaries for diagnostics."""
    cf_summary = summarize_recbole_parquet(RECBOLE_CF_PATH, "CF", catalog_info)
    seq_summary = summarize_recbole_parquet(
        RECBOLE_SEQ_PATH, "Sequential", catalog_info
    )
    candidate_summary = pd.concat([cf_summary, seq_summary], ignore_index=True)
    candidate_summary.to_csv(OUTPUT_DIR / "recbole_candidate_summary.csv", index=False)
    candidate_summary.to_parquet(
        OUTPUT_DIR / "recbole_candidate_summary.parquet", index=False
    )

    unified = pd.read_csv(RECBOLE_UNIFIED_SUMMARY_PATH).drop_duplicates()
    standalone = unified.loc[
        unified["model_type"].isin(["CF", "Sequential"])
        & unified["eval_method"].eq("standalone")
        & unified["k"].eq(10)
    ].copy()
    standalone = standalone.sort_values("ndcg", ascending=False)
    standalone.to_csv(OUTPUT_DIR / "recbole_standalone_summary.csv", index=False)
    return candidate_summary


def pairwise_jaccard_distance(lists: Sequence[Sequence[str]]) -> float:
    """Mean pairwise Jaccard distance over recommendation lists."""
    distances: list[float] = []
    for left, right in itertools.combinations(lists, 2):
        left_set = set(left)
        right_set = set(right)
        union = left_set | right_set
        distances.append(
            0.0 if not union else 1.0 - (len(left_set & right_set) / len(union))
        )
    return float(np.mean(distances)) if distances else np.nan


def pairwise_position_distance(lists: Sequence[Sequence[str]], k: int = 10) -> float:
    """Mean pairwise position mismatch distance over top-k lists."""
    distances: list[float] = []
    for left, right in itertools.combinations(lists, 2):
        mismatches = 0
        for idx in range(k):
            left_item = left[idx] if idx < len(left) else "__missing__"
            right_item = right[idx] if idx < len(right) else "__missing__"
            mismatches += int(left_item != right_item)
        distances.append(mismatches / k)
    return float(np.mean(distances)) if distances else np.nan


def pairwise_exact_list_agreement(lists: Sequence[Sequence[str]]) -> float:
    """Pairwise fraction of trials with identical top-k lists."""
    agreements = [
        float(list(left) == list(right))
        for left, right in itertools.combinations(lists, 2)
    ]
    return float(np.mean(agreements)) if agreements else np.nan


def load_stability_trials() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load and validate stability trial and aggregate files."""
    trial_parts = [pd.read_parquet(path) for path in STABILITY_TRIAL_PATHS]
    trials = pd.concat(trial_parts, ignore_index=True)
    assert len(trials) == 25200, f"Unexpected stability row count: {len(trials)}"
    duplicate_keys = trials.duplicated(
        ["model", "prompt_idx", "temperature", "trial_idx"]
    )
    assert not duplicate_keys.any(), "Duplicate stability model/prompt/temp/trial rows"
    assert not trials["model"].str.contains("gemini", case=False, na=False).any()

    aggregate_parts = [pd.read_parquet(path) for path in STABILITY_AGGREGATE_PATHS]
    aggregates = pd.concat(aggregate_parts, ignore_index=True)
    return trials, aggregates


def build_stability_prompt_summary(
    scored_trials: pd.DataFrame,
    catalog_info: dict[str, Any] | None = None,
) -> pd.DataFrame:
    """Compute prompt-level stability quality and list-distance metrics."""
    rows: list[dict[str, Any]] = []
    group_cols = ["model", "model_display", "prompt_idx", "temperature"]
    for group_key, group in scored_trials.groupby(group_cols):
        model, model_display, prompt_idx, temperature = cast(
            "tuple[str, str, int, float]", group_key
        )
        raw_lists = [canonical_topk(as_list(items)) for items in group["pred_items"]]
        strict_lists = [
            canonical_topk(as_list(items)) for items in group["strict_pred_items"]
        ]
        raw_ndcg = group["raw_ndcg@10"].to_numpy(dtype=float)
        strict_ndcg = group["strict_ndcg@10"].to_numpy(dtype=float)
        record = {
            "model": model,
            "model_display": model_display,
            "prompt_idx": int(prompt_idx),
            "temperature": float(temperature),
            "n_trials": len(group),
            "raw_ndcg@10_mean": float(np.nanmean(raw_ndcg)),
            "raw_ndcg@10_std": float(np.nanstd(raw_ndcg, ddof=1)),
            "strict_ndcg@10_mean": float(np.nanmean(strict_ndcg)),
            "strict_ndcg@10_std": float(np.nanstd(strict_ndcg, ddof=1)),
            "candidate_valid_rate@10_mean": float(
                group["candidate_valid_rate@10"].mean()
            ),
            "raw_jaccard_distance@10": pairwise_jaccard_distance(raw_lists),
            "strict_jaccard_distance@10": pairwise_jaccard_distance(strict_lists),
            "raw_list_position_distance@10": pairwise_position_distance(raw_lists),
            "strict_list_position_distance@10": pairwise_position_distance(
                strict_lists
            ),
            "raw_exact_list_agreement@10": pairwise_exact_list_agreement(raw_lists),
            "strict_exact_list_agreement@10": pairwise_exact_list_agreement(
                strict_lists
            ),
        }
        if catalog_info is not None:
            for prefix, lists in {"raw": raw_lists, "strict": strict_lists}.items():
                set_metrics = recommendation_set_metrics(
                    lists,
                    catalog_info["catalog_size"],
                    catalog_info["popularity_by_canonical"],
                    10,
                )
                for metric_name, value in set_metrics.items():
                    record[f"{prefix}_{metric_name}"] = value
        record["raw_ndcg@10_cv"] = (
            record["raw_ndcg@10_std"] / record["raw_ndcg@10_mean"]
            if record["raw_ndcg@10_mean"]
            else np.nan
        )
        record["strict_ndcg@10_cv"] = (
            record["strict_ndcg@10_std"] / record["strict_ndcg@10_mean"]
            if record["strict_ndcg@10_mean"]
            else np.nan
        )
        rows.append(record)
    return pd.DataFrame.from_records(rows)


def build_stability_summary(prompt_summary: pd.DataFrame) -> pd.DataFrame:
    """Aggregate prompt-level stability summaries by model and temperature."""
    rows: list[dict[str, Any]] = []
    value_cols = [
        "raw_ndcg@10_mean",
        "strict_ndcg@10_mean",
        "raw_ndcg@10_std",
        "strict_ndcg@10_std",
        "raw_ndcg@10_cv",
        "strict_ndcg@10_cv",
        "candidate_valid_rate@10_mean",
        "raw_jaccard_distance@10",
        "strict_jaccard_distance@10",
        "raw_list_position_distance@10",
        "strict_list_position_distance@10",
        "raw_exact_list_agreement@10",
        "strict_exact_list_agreement@10",
        "raw_item_coverage@10",
        "strict_item_coverage@10",
        "raw_avg_popularity@10",
        "strict_avg_popularity@10",
        "raw_catalog_match_rate@10",
        "strict_catalog_match_rate@10",
    ]
    for group_key, group in prompt_summary.groupby(
        ["model", "model_display", "temperature"]
    ):
        model, model_display, temperature = cast("tuple[str, str, float]", group_key)
        record = {
            "model": model,
            "model_display": model_display,
            "temperature": float(temperature),
            "n_prompts": group["prompt_idx"].nunique(),
            "n_trials": int(group["n_trials"].sum()),
        }
        for col in value_cols:
            values = group[col].to_numpy(dtype=float)
            record[col] = float(np.nanmean(values))
            record[f"{col}_prompt_std"] = float(np.nanstd(values, ddof=1))
        for col in ["strict_ndcg@10_mean", "strict_jaccard_distance@10"]:
            mean, low, high = bootstrap_mean_ci(
                group[col].to_numpy(dtype=float), RNG, n_boot=2000
            )
            record[f"{col}_ci_mean"] = mean
            record[f"{col}_ci_low"] = low
            record[f"{col}_ci_high"] = high
        rows.append(record)
    return sort_model_frame(pd.DataFrame.from_records(rows))


# Model ordering for stability plots. Non-FT open-weight models keep the same
# color/marker index as the RQ2 figure; Qwen2.5-7B-FT takes the unused sixth
# Okabe-Ito slot so its line is visually distinct from any RQ2 model.
RQ2_OPEN_WEIGHT_MODELS = OPEN_WEIGHT_MODELS
STABILITY_OPEN_WEIGHT_ORDER = [
    "Llama-3.3-70B",
    "Gemma-2-9B",
    "Llama-3.1-8B",
    "Qwen2.5-7B",
    "Llama-3.2-3B",
    "Qwen2.5-7B-FT",
]
# Cap legend entries per row so the legend rows stay narrow enough to span
# both subfigures at \columnwidth.
STABILITY_LEGEND_MAX_PER_ROW = 3
# Local rcParams overrides for the RQ4 figure: bumped font and marker sizes
# so the two-panel layout reads at least as legibly as the RQ2 single-panel
# figure once both are placed at \columnwidth in the LaTeX document.
STABILITY_RC_PARAMS = {
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "legend.title_fontsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "lines.linewidth": 1.4,
    "lines.markersize": 5.0,
}


def _stability_style(model: str) -> tuple[str, str]:
    """Return (color, marker) matching the RQ2 figure assignment by model name."""
    if model in PROPRIETARY_MODELS:
        idx = PROPRIETARY_MODELS.index(model)
    elif model == "Qwen2.5-7B-FT":
        idx = 5
    elif model in RQ2_OPEN_WEIGHT_MODELS:
        idx = RQ2_OPEN_WEIGHT_MODELS.index(model)
    else:
        idx = 0
    return OKABE_ITO[idx], RQ2_MARKERS[idx]


def _row_major_for_column_layout(handles, ncol):
    """Reorder handles so a column-first legend reads row-first visually."""
    n = len(handles)
    if n == 0:
        return handles
    nrow = (n + ncol - 1) // ncol
    out = []
    for c in range(ncol):
        for r in range(nrow):
            idx = r * ncol + c
            if idx < n:
                out.append(handles[idx])
    return out


def save_stability_figure(
    stability_summary: pd.DataFrame, suffix: str = "full"
) -> None:
    """Save RQ4 temperature-sensitivity figure.

    Two side-by-side panels: NDCG@10 (A, left) and Jaccard distance@10
    (B, right) versus decoding temperature. Styling matches the RQ2
    candidate-pool figure: proprietary models use solid lines with filled
    markers, open-weight models use dashed lines with hollow markers, and
    color and marker for each model match the RQ2 figure by name.
    Qwen2.5-7B-FT, absent from RQ2, takes the unused sixth Okabe-Ito
    color. Legends sit below the panels, capped at three entries per row,
    and read left-to-right first.
    """
    df = sort_model_frame(stability_summary)
    available = set(df["model_display"].unique())
    prop_models = [m for m in PROPRIETARY_MODELS if m in available]
    open_models = [m for m in STABILITY_OPEN_WEIGHT_ORDER if m in available]

    with plt.rc_context(STABILITY_RC_PARAMS):
        fig, axes = plt.subplots(1, 2, figsize=(4.6, 2.6), sharex=True)
        panels = [
            (axes[0], "strict_ndcg@10_mean", "NDCG@10", "A"),
            (axes[1], "strict_jaccard_distance@10", "Jaccard distance@10", "B"),
        ]

        def _plot_group(ax, y_col, models, linestyle, *, hollow):
            for model in models:
                sub = df.loc[df["model_display"].eq(model)].sort_values("temperature")
                if sub.empty:
                    continue
                color, marker = _stability_style(model)
                ax.plot(
                    sub["temperature"],
                    sub[y_col],
                    color=color,
                    linestyle=linestyle,
                    marker=marker,
                    markerfacecolor="white" if hollow else color,
                    markeredgecolor=color,
                    markeredgewidth=1.2,
                    label=model,
                )

        for ax, y_col, ylabel, tag in panels:
            _plot_group(ax, y_col, prop_models, linestyle="-", hollow=False)
            _plot_group(ax, y_col, open_models, linestyle="--", hollow=True)
            ax.set_xlabel("Temperature")
            ax.set_ylabel(ylabel)
            ax.set_xticks([0.0, 0.5, 1.0, 2.0])
            _minimal_axes(ax)
            ax.set_title(rf"\textbf{{{tag}}}", loc="left", fontsize=13, pad=8)
        axes[0].set_yticks([0.00, 0.04, 0.08, 0.12])
        axes[0].set_ylim(0.0, 0.13)
        axes[1].set_ylim(0.0, 1.0)

        # Tight subplot packing without clipping panel titles.
        fig.tight_layout(pad=0.4, w_pad=1.0, h_pad=0.6)

        def _handle(model, linestyle, *, hollow):
            color, marker = _stability_style(model)
            return Line2D(
                [],
                [],
                color=color,
                linestyle=linestyle,
                marker=marker,
                markerfacecolor="white" if hollow else color,
                markeredgecolor=color,
                markeredgewidth=1.2,
                label=model,
            )

        prop_handles = [_handle(m, "-", hollow=False) for m in prop_models]
        open_handles = [_handle(m, "--", hollow=True) for m in open_models]

        prop_ncol = min(STABILITY_LEGEND_MAX_PER_ROW, len(prop_handles))
        open_ncol = min(STABILITY_LEGEND_MAX_PER_ROW, len(open_handles))
        prop_rows = (len(prop_handles) + prop_ncol - 1) // prop_ncol

        prop_handles = _row_major_for_column_layout(prop_handles, prop_ncol)
        open_handles = _row_major_for_column_layout(open_handles, open_ncol)

        leg1 = fig.legend(
            handles=prop_handles,
            title=r"\textit{Proprietary}",
            loc="upper center",
            bbox_to_anchor=(0.5, -0.005),
            ncol=prop_ncol,
            borderpad=0.2,
            handlelength=1.6,
            handletextpad=0.4,
            columnspacing=1.4,
            frameon=False,
        )
        open_y = -0.05 - 0.15 * prop_rows
        leg2 = fig.legend(
            handles=open_handles,
            title=r"\textit{Open-weight}",
            loc="upper center",
            bbox_to_anchor=(0.5, open_y),
            ncol=open_ncol,
            borderpad=0.2,
            handlelength=1.6,
            handletextpad=0.4,
            columnspacing=1.4,
            frameon=False,
        )

        ipy_display(fig)
        for ext in ["png", "pdf"]:
            fig.savefig(
                FIGURE_DIR / f"figure_rq4_temperature_sensitivity_{suffix}.{ext}",
                bbox_extra_artists=(leg1, leg2),
                bbox_inches="tight",
                pad_inches=0.05,
            )
        plt.close(fig)


def save_stability_representative_figure(stability_summary: pd.DataFrame) -> None:
    """Save a compact representative-model version of RQ4 figure."""
    representative = [
        "Claude Opus 4.6",
        "GPT-5.2",
        "GPT-4.1-mini",
        "Qwen2.5-7B-FT",
        "Gemma-2-9B",
        "Llama-3.3-70B",
    ]
    compact = stability_summary.loc[
        stability_summary["model_display"].isin(representative)
    ].copy()
    save_stability_figure(compact, suffix="representative")

## 1. Load prompts, catalog metadata, and deterministic LLM evaluations

In [ ]:
ensure_dirs()

# Paper figure style: serif font with LaTeX backend, minimal axes, paper-sized.
plt.rcParams.update(
    {
        "text.usetex": True,
        "text.latex.preamble": r"\usepackage{lmodern}",
        "font.family": "serif",
        "font.size": 8,
        "axes.labelsize": 8,
        "axes.titlesize": 8,
        "legend.fontsize": 7,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "axes.linewidth": 0.6,
        "axes.edgecolor": "#333333",
        "axes.labelcolor": "#222222",
        "xtick.color": "#333333",
        "ytick.color": "#333333",
        "xtick.major.width": 0.5,
        "ytick.major.width": 0.5,
        "xtick.major.size": 3.0,
        "ytick.major.size": 3.0,
        "lines.linewidth": 1.1,
        "lines.markersize": 3.2,
        "legend.frameon": False,
        "legend.handlelength": 1.8,
        "legend.borderaxespad": 0.2,
        "legend.columnspacing": 1.0,
        "legend.labelspacing": 0.3,
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
    }
)


def _minimal_axes(ax) -> None:
    """Apply minimal axes: hide top/right spines, faint horizontal grid."""
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.grid(True, axis="y", color="#cccccc", linewidth=0.4, alpha=0.6)
    ax.set_axisbelow(True)


prompt_info = load_prompt_info()
catalog_info = load_catalog_info()

llm = load_deterministic_llm()
llm.to_parquet(OUTPUT_DIR / "deterministic_llm_clean.parquet", index=False)

print(f"Clean deterministic LLM rows: {len(llm):,}")
print(f"Catalog items: {catalog_info['catalog_size']:,}")
print(llm.groupby(["model", "n_candidates"]).size().to_string())

## 2. Score deterministic LLM outputs

In [ ]:
llm_scored = score_rows(llm, prompt_info, catalog_info=catalog_info)
llm_scored.to_parquet(OUTPUT_DIR / "deterministic_llm_scored.parquet", index=False)

deterministic_summary = summarize_evaluation(
    llm_scored,
    ["model", "model_display", "model_type", "n_candidates"],
    catalog_info=catalog_info,
)
deterministic_summary = sort_model_frame(deterministic_summary)
deterministic_summary.to_csv(OUTPUT_DIR / "deterministic_summary.csv", index=False)
deterministic_summary.to_parquet(
    OUTPUT_DIR / "deterministic_summary.parquet", index=False
)

deterministic_summary[
    [
        "model_display",
        "n_candidates",
        "raw_ndcg@10",
        "strict_ndcg@10",
        "paper_ndcg@10",
        "paper_hit_rate@10",
        "paper_item_coverage@10",
        "paper_avg_popularity@10",
    ]
].head(20)

## RQ1 — LLM rerankers under zero-shot and semantic@250

The `∗` marker denotes NDCG@10 significantly better than EASE under a paired Wilcoxon signed-rank test with Holm correction (`p < 0.05`). EASE is the semantic@250 candidate-scoring baseline from the RecBole CF output.


In [ ]:
significance_c0 = wilcoxon_vs_ease(llm_scored, "c0")
significance_c250 = wilcoxon_vs_ease(llm_scored, "c250")
significance_all = pd.concat(
    [significance_c0.assign(setting="c0"), significance_c250.assign(setting="c250")],
    ignore_index=True,
)
significance_all.to_csv(OUTPUT_DIR / "rq1_wilcoxon_vs_ease.csv", index=False)

rq1_zero_shot, rq1_zero_stats = build_rq1_table(
    deterministic_summary, significance_c0, "c0"
)
rq1_semantic250, rq1_semantic250_stats = build_rq1_table(
    deterministic_summary,
    significance_c250,
    "c250",
)

rq1_zero_shot.to_csv(OUTPUT_DIR / "table_rq1_zero_shot_llms.csv", index=False)
rq1_semantic250.to_csv(OUTPUT_DIR / "table_rq1_semantic250_llms.csv", index=False)
pd.concat([rq1_zero_stats, rq1_semantic250_stats], ignore_index=True).to_csv(
    OUTPUT_DIR / "table_rq1_llms_numeric.csv",
    index=False,
)

print("Zero-shot")
ipy_display(rq1_zero_shot)
print("Semantic@250")
ipy_display(rq1_semantic250)

## RQ2 — Candidate-pool size and full-catalog access

In [ ]:
rq2_table = build_rq2_table(deterministic_summary)
rq2_table = rq2_table.drop(columns=["candidate_order"], errors="ignore")
rq2_table.to_csv(OUTPUT_DIR / "figure_rq2_candidate_pool_table.csv", index=False)
save_rq2_line_figure(rq2_table)

rq2_pivot = rq2_table.pivot_table(
    index="model_display",
    columns="Candidate pool",
    values=PRIMARY_METRIC,
    aggfunc="first",
).reindex(index=MODEL_ORDER, columns=["Zero-shot", "250", "500", "1000", "All"])
rq2_pivot.round(4)

## RQ3 — Retriever effects

`CandR@250` and `Oracle NDCG@10` are computed directly from the three candidate-pool prompt files. NDCG columns are filled for every model in `RQ3_RERANKER_MODELS`; semantic@250 comes from the deterministic run, while EASE@250 and SASRec@250 come from `data/output/evaluation_results_rq3_*.parquet`.


In [ ]:
recbole_candidate_summary = load_recbole_summaries(catalog_info)
rq3_table, rq3_llm_summary = build_rq3_table(
    prompt_info, deterministic_summary, catalog_info
)
ipy_display(rq3_table.round(4))

ndcg_cols = [
    col
    for col in rq3_table.columns
    if col.endswith("NDCG@10") and col != "Oracle NDCG@10"
]
missing = rq3_table.loc[
    rq3_table[ndcg_cols].isna().any(axis=1), ["Retriever", *ndcg_cols]
]
if not missing.empty:
    print(
        f"Some RQ3 LLM reranking cells are still missing. Generate data/output/{RQ3_RESULT_GLOB} "
        "with scripts/04_evaluation.ipynb in RQ3 mode for the missing model/retriever combinations."
    )
    ipy_display(missing)

## RQ4 — Temperature sensitivity and list stability

In [ ]:
stability_trials, stability_aggregates = load_stability_trials()
stability_aggregates.to_parquet(
    OUTPUT_DIR / "stability_aggregated_source.parquet", index=False
)

stability_scored = score_rows(
    stability_trials,
    prompt_info,
    catalog_info=catalog_info,
    fixed_prompt_source="c250",
)
stability_scored.to_parquet(
    OUTPUT_DIR / "stability_per_trial_scored.parquet", index=False
)

stability_prompt_summary = build_stability_prompt_summary(
    stability_scored, catalog_info
)
stability_prompt_summary.to_csv(
    OUTPUT_DIR / "stability_prompt_summary.csv", index=False
)
stability_prompt_summary.to_parquet(
    OUTPUT_DIR / "stability_prompt_summary.parquet", index=False
)

stability_summary = build_stability_summary(stability_prompt_summary)
stability_summary.to_csv(OUTPUT_DIR / "stability_summary.csv", index=False)
stability_summary.to_parquet(OUTPUT_DIR / "stability_summary.parquet", index=False)

save_stability_figure(stability_summary, suffix="full")
save_stability_representative_figure(stability_summary)

stability_summary[
    [
        "model_display",
        "temperature",
        "strict_ndcg@10_mean",
        "strict_jaccard_distance@10",
        "strict_list_position_distance@10",
        "strict_exact_list_agreement@10",
    ]
].head(20)

## Output files

Key outputs are written under `data/output/paper/` and `data/output/paper/figures/`.


In [ ]:
print(f"Tables and summaries: {OUTPUT_DIR.relative_to(ROOT)}/")
print(f"Figures: {FIGURE_DIR.relative_to(ROOT)}/")
print("\nRQ outputs:")
for path in [
    OUTPUT_DIR / "table_rq1_zero_shot_llms.csv",
    OUTPUT_DIR / "table_rq1_semantic250_llms.csv",
    OUTPUT_DIR / "figure_rq2_candidate_pool_table.csv",
    OUTPUT_DIR / "table_rq3_retriever_effect.csv",
    OUTPUT_DIR / "table_rq3_retriever_effect_long.csv",
    OUTPUT_DIR / "stability_summary.csv",
    FIGURE_DIR / "figure_rq2_candidate_pool_ndcg.png",
    FIGURE_DIR / "figure_rq4_temperature_sensitivity_full.png",
    FIGURE_DIR / "figure_rq4_temperature_sensitivity_representative.png",
]:
    print("-", path.relative_to(ROOT))